# Installation
First install the necessary packages.

In [ ]:
!pip install transformers[torch]==4.28.0
!pip install SentencePiece
!pip install wandb -q
!pip install functorch

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


# Import and Login
To track changes, we use wandb. This might be optional, but I don't know about that, so if you don't have an account, make one!

In [ ]:
import json
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, GenerationConfig
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb
import re
import os
from google.colab import files
import shutil

from torch import cuda
device = 'cuda'
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

!wandb login
!wandb status

wandb: Currently logged in as: boothster. Use `wandb login --relogin` to force relogin
Current Settings
{
  "_extra_http_headers": null,
  "api_key": null,
  "base_url": "https://api.wandb.ai",
  "entity": null,
  "git_remote": "origin",
  "ignore_globs": [],
  "project": null,
  "root_dir": null,
  "section": "default"
}


# Custom Classes

## Dataset
To load our data, we use a specific data structure which we can parse in any way we choose. For this
model, we simply tokenize data and return required keys

In [ ]:
#Custom Dataset, to ensure we can load exactly what we need from the dataset
class TokenizedDataset(Dataset):

    def __init__(self,
                 dataframe,
                 tokenizer,
                 dataset_config):
        self.context = dataframe.context
        self.target = dataframe.target
        self.config = dataset_config
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.context)

    #Get an encoded item from the dataset
    def __getitem__(self, index):
        source = self.tokenizer.encode_plus(
            self.context[index],
            max_length=self.config['padding']['sel_len'],
            padding="max_length", truncation=True,
            return_tensors='pt')
        target = self.tokenizer.encode_plus(
            self.target[index],
            max_length=self.config['padding']['inf_len'],
            padding="max_length", truncation=True,
            return_tensors='pt')

        y = target['input_ids'].to(dtype=torch.long).squeeze(0)
        y_ids = y[:-1].contiguous()
        lm_labels = y[1:].clone().detach()
        lm_labels[y[1:] == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': source['input_ids'].to(dtype=torch.long).squeeze(0),
            'attention_mask': source['attention_mask'].to(dtype=torch.long).squeeze(0),
            'decoder_input_ids': y_ids.to(dtype=torch.long).squeeze(0),
            'labels': lm_labels.to(dtype=torch.long).squeeze(0)}

#Custom Dataset, to ensure we can load exactly what we need from the dataset
class RawDataset(Dataset):

    def __init__(self, dataframe):
        self.context = dataframe.context
        self.target = dataframe.target

    def __len__(self):
        return len(self.context)

    def __getitem__(self, index):
        return {
            'context': self.context[index],
            'target': self.target[index]
        }

    # Process dataframe to get a custom dictionary for our needs
    def get_context_dictionary(self, batch_size):
        loader = DataLoader(self, batch_size = batch_size)
        context_array_split = []
        for _, data in enumerate(loader):
            context = data['context']
            # Get question and hypothesis from context
            for i in range(len(context)):
                index = _ * batch_size + i
                pattern = "hypothesis:.+?:"
                hypothesis = re.findall(pattern, context[i])[0][:-7]
                hypothesis = re.sub("hypothesis: ", "", hypothesis)
                pattern = "question:.+?:"
                question = re.findall(pattern, context[i])[0][:-12]
                question = re.sub("question: ", "", question)

                # intitialise dictionary
                context_array_split.append(
                    {"index": index,
                     "hypothesis": hypothesis,
                     "preface": f"{question} {hypothesis}",
                     "beams": [{"length_of_context": 0,
                                "context": [],
                                "score": 0,
                                "proof": "$proof$ = "}],
                     "done": False
                })

                # find sentences in the context
                selected_sents = re.findall(r'sent\d{1,2}', context[i])
                for sent in selected_sents:
                    pattern = sent + ":.+?:|" + sent + ":.+?$"
                    sent_text = re.findall(pattern, context[i])[0]
                    if sent_text[-1] == ":":
                        sent_text = sent_text[:-7]
                    # Add initialise beams with each item in the context
                    context_array_split[index][
                        "beams"][0][
                            "context"].append(sent_text)
                    context_array_split[index][
                        "beams"][0][
                            "length_of_context"] += 1
        return context_array_split

## Logit Processor
Calculating the MLE loss is subotimal for our purposes, so we define a custom logit processor for generation, to inform likelihood estimates targeted towards the hypothesis. We also define a similar logit processor for scoring selections, biased towards pairs which share more words than ones which share fewer. Finally, we define a logit processor for scoring one step proofs, biased towards similar facts and inferences towards a hypothesis

In [ ]:
from transformers.generation.logits_process import LogitsProcessor
import math

class DifferenceLogitProcessor(LogitsProcessor):
    r"""
    ['LogitsProcessor']
    Applies a bonus to tokens in the provided goal that are not already
    generated. Similar to the encoder repetition penalty.

    We provide a "goal" sequance (which is different from a perfect output
    sequence) to inform generation of intermediate sequences. We want to apply
    a bonus to scores of tokens in that sequence, decreasing after a threshold,
    at which point we likely want some novel generation to be more like the
    perfect output.

    This decrease can't be continuous, as when we promote multiple sequences,
    the result of a linear decrease is to choose a word from each sequence in
    turn, rather than compose phrases from each one.

    Args:
        batches ('int'):
            Number of batches processed, used to store lengths of inter-
            sections, across all batches, and all beams for each in the
            batch.
        num_beams ('int'):
            Number of beams when beam search is activated, used to repeat
            the goal sequence
        goal ('torch.LongTensor'):
            Tokenized goal string. We want to generate a sequence which is
            similar to this, but typically only has <50% of it's tokens.
            We control this threshold with the penalties and bonus.

            e.g. prompt:  "comets orbit the sun, comet orbits are elliptcal"
                 perfect: "comets orbit the sun elliptically"
                 goal:    "comets, which orbit the sun elliptically, are
                           made of ice."
            We first process the goal to be a bit more specific to the prompt,
            but this is the gist of it.
        params ('dict'):
            max_bonus ('float'):
                bonus to apply to tokens from the goal that have not yet been
                generated.
            min_bonus ('float'):
                min bonus to apply to tokens from the goal that have not yet
                been generated, after the number of generated tokens reaches the
                threshold, the actualy applied bonus will decrease towards min
            threshold ('float'):
                threshold after which it's less important that we generate tokens
                which are in the goal sequence.
            steps ('int'):
                the number of steps to group tokens in, and calculate a common
                bonus for.
        version ('string'):
            used to separate between possible goal facts. Only used to apply
            a penalty rather than a bonus for tokens shared between the goal
            and generated tokens. This is useful as some induction rules
            require us to substitute tokens common to both facts with tokens
            relevant tokens in either fact, while other rules require us to
            use tokens common to both facts to find a property of that thing.

            e.g. the difference between "comets orbit the sun", with:
                case 1."comet orbits are elliptcal", or
                case 2."orbits around the sun are elliptical".
                -> "comets orbit the sun elliptically".
            With the exception of "orbits", in case 1 want to use repeated
            tokens "comet", in case 2 we want to exclude them "the", "sun".
            We can use beam groups to find scores for either case.
    """
    def __init__(self, batches: int, num_beams: int, goal: torch.LongTensor,
                 params: dict):

        # Reshape goal to match the beam size, and initialise lengths of
        # intersections between generated tokens and the goal, for this beam
        self.get_goal(goal, num_beams)

        magnitude = torch.FloatTensor([params['magnitude']]).to(goal.device)
        decay = torch.FloatTensor([params['decay']]).to(goal.device)
        self.steps = params['steps']
        self.get_thresholds(params['threshold'], goal.device,
                            num_beams * batches, magnitude, decay)

    # Get data associated with the goal sequence at initialisation
    def get_goal(self, goal, num_beams):
        self.goal = goal.repeat_interleave(num_beams, dim=0)
        self.goal_mask = (self.goal > 3) & (self.goal < 32000)
        self.initial_length = torch.sum(self.goal_mask, dim=1)
        initial_mask = self.initial_length > 0

    # Return a range of values from [inf, -inf], decreasing linearly proportional
    # to decay, and crossing the point (threshold, 1), given a range from
    # [threshold, 1]
    def threshold_decay(self, x, decay, threshold):
        decay = decay.expand(self.steps)
        return (decay/(1-threshold)) * (threshold-x) + 1

    # Transform the above into values between [magnitude, 0], where magnitude is
    # modified, and crossing the point (magnitude, threshold) where magnitude is
    # as provided in the paramaters dictionary
    def sigmoid_decay(self, x, magnitude, decay, threshold):
        return magnitude * torch.sigmoid(
            self.threshold_decay(x, decay, threshold))


    # Get thresholds for decay at initialisation
    def get_thresholds(self, threshold, device, shape, magnitude, decay):
        bound = torch.linspace(threshold, 1, self.steps).to(device)

        # If the threshold is 1, we have division by 0
        if threshold < 1:
            threshold = torch.FloatTensor([threshold]).to(device)
            sigmoid_threshold = self.sigmoid_decay(
                threshold, magnitude, decay, threshold)
            magnitude = torch.square(magnitude) / sigmoid_threshold
            bonus = self.sigmoid_decay(
                bound, magnitude, decay, threshold)
            self.bonus = bonus.unsqueeze(0).repeat(shape, 1)
            self.bound = bound.unsqueeze(0).repeat(shape, 1)
            print(self.bound.shape, bound.unsqueeze(0))
        else:
            self.bonus = bonus.unsqueeze(0).repeat(shape, self.steps)
            self.bound = bound.unsqueeze(0).repeat(shape, 1)

    def __call__(self, input_ids: torch.LongTensor,
                 scores: torch.FloatTensor) -> torch.FloatTensor:
        r"""
        Apply the logit processor, by giving a bonus to scores for tokens
        which appear in the goal sequence. If the last generated token was
        in the goal sequence, we want to remove it from the goal sequence
        first. We also give a smaller bonus after a threshold is reached.
            Args:
                input_ids ('int'):
                    Ids of all tokens generated across all beams in this group.
                scores ('int'):
                    Scores for potential next tokens in each beam in the group.
        """
        # Find the intersection of generated tokens and the goal
        intersection = torch.vmap(lambda t1, t2: torch.isin(
            t1,t2))(self.goal, input_ids)

        # Find the length of currently intersecting tokens, provided they
        # are not masked (padded) tokens, then update the proportion and mask
        non_pad_intersect = intersection & self.goal_mask
        intersect_length = torch.sum(non_pad_intersect,dim=1)
        proportion = intersect_length / self.initial_length

        # Apply proportionality to the bonus
        bonus = self.decay(proportion).unsqueeze(1)
        bonus = bonus.expand(self.goal.shape[1], self.goal.shape[2])
        # Apply the bonus to the relevant scores
        bonus = torch.where(self.goal_mask, bonus, 1)
        scores_ = torch.gather(scores, 1, self.goal)
        scores_ = torch.where(scores_ < 0, scores_ / bonus, scores_ * bonus)
        scores.scatter_(1, self.goal, scores_)
        return scores

    def decay(self, proportion):
        r"""
        Apply a logarithmic decay to the bonus, given a proportion.

            Args:
                proportion ('torch.FloatTensor'):
                    Used to calculate logarithmic decay. We first
                    calculate which bound the proportion belongs to, where the
                    bound size decreases logarithmicly as proportion increases.
                    Then we calculate the linear decay, mapping from the
                    logarithmic bound to the linear decay.
        """
        # Repeat proportion: x so we can compare it to the thresholds in
        # parallel, and find the lowest threshold greater than x
        proportion = proportion.unsqueeze(1).repeat_interleave(
            self.steps, dim=1)
        # find the highest bonus where the proportion of intersected facts is
        # below the threshold
        current_bound = torch.where(
            proportion <= self.bound, self.bonus, float("-inf"))
        bonus = torch.max(current_bound, dim=1)
        return bonus.values


## Trainer
This is the substantial portion of the model, where we overwrite the evaluation function, to produce entailment trees, as the model will not produce them in one step.
In short, we first select facts as they are scored by the model, then we compose a hyper-parameter number of selections, based on the highest scoring facts.
Next, we produce three inferences for each selection, ideally, one will be similar to each fact that produces it, and one will be a mesh of both.
Finally, we score the entire selection and inference, as a readable sentence, to decide which inference and selection has the highest score.

We then iterate, keeping a hyper-parameter number of beams, or partial entailment trees, as given by these high scoring entailments.

In [ ]:
#Custom trainer, overwrites the evaluate function
class CustomTrainer(Trainer):
    def  __init__(self,
                  trainer_model,
                  args: TrainingArguments,
                  trainer_train_dataset: TokenizedDataset,
                  trainer_eval_dataset: Dataset,
                  trainer_tokenizer: T5Tokenizer,
                  trainer_device: str,
                  trainer_config):
        Trainer.__init__(self,
                         model = trainer_model,
                         args = args,
                         train_dataset = trainer_train_dataset,
                         eval_dataset = TokenizedDataset(
                              trainer_eval_dataset,
                              trainer_tokenizer,
                              trainer_config),
                         tokenizer=trainer_tokenizer
                         )
        self.context: dict = {}
        self.device = trainer_device
        self.config = trainer_config
        self.raw_eval = RawDataset(trainer_eval_dataset)

    # Evaluate as a trainer usually would, then calculate an entailment tree, and write it to a file
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix: str = 'eval'):
        root = self.config['directory']['runs']
        method = f"{root}/{self.config['directory']['method']}/"
        version = f"{method}/{self.config['directory']['version']}/"
        trees = f"{version}/trees"
        runs = os.listdir(trees)
        step = int(self.state.global_step)
        step = str(step * self.config['batches']['train'])
        if not step in " ".join(runs):
            torch.cuda.empty_cache()
            score, predictions = self.validate()
            if self.config['logging']['save_file']:
                write = f'{trees}/steps{step}-{str(int(score))}'
                with open(f'{write}.tsv', 'w') as write_tsv:
                    write_tsv.write(predictions.to_csv(sep='\t', index=False))
        metrics = Trainer.evaluate(self, eval_dataset=self.eval_dataset,
                                   ignore_keys=ignore_keys,
                                   metric_key_prefix=metric_key_prefix)
        return metrics

    def validate(self):
        with torch.no_grad():
            self.context = self.raw_eval.get_context_dictionary(
                self.config['batches']['test'])[:8]
            for i in range(self.config['search']['iterations']):
                self.gen_selections()
                self.gen_inferences(i)

                if i == 0:
                    print(self.state.global_step,
                        "(iteration, enatailment beams, inference beams): ",
                        end = "")
                beams = sum([len(question['beams'])
                for question in self.context])
                print(((i + 1), beams, beams * self.config[
                    'search']['infer_beams']), end=", ")

            probabilities = [question["beams"][0]["score"]
                             for question in self.context]
            predictions = pd.DataFrame([question["beams"][0]["proof"]
                                        for question in self.context])
            print("")
            torch.cuda.empty_cache()
            return -sum(probabilities), predictions

    # Get the pretokenized pair of facts, along with the full text
    # to feed into the model, for scoring pairs.
    def get_pair_strings(self):
        selections = []
        left = []
        right = []
        hypothesis = []
        pattern = r'sent\d{1,2}: |int\d{1,2}: '
        for question in self.context:
            if not question["done"]:
                for beam in question["beams"]:
                    fact_string = '. '.join([re.sub(pattern, '', sent)
                                            for sent in beam['context']])
                    a, b = beam["selection"]
                    selections.append({
                        "context": f"correlate: {question['preface']}. {fact_string}.",
                        "target": " and ".join([f"{re.sub(pattern, '', a)}",
                                                f"{re.sub(pattern, '', b)}."])})
                    left.append(re.sub(pattern, '', a))
                    right.append(re.sub(pattern, '', b))
                    hypothesis.append(question['hypothesis'])
        return selections, (left,right), hypothesis

    # Generate probabilities for a given selection. Return these
    # probabilities, along with the pair of facts which produced them
    def score_pairs(self) -> torch.Tensor:
        selections, (left, right), hypothesis = self.get_pair_strings()
        avg = []
        if selections:
            # Tokenize
            n_questions = len(selections)
            for _i in range(0, n_questions, self.config['batches']['test']):
                i_ = _i+self.config['batches']['test']
                if i_ > n_questions:
                    i_ = n_questions
                # Tokenize our selected facts
                a = self.tokenizer.batch_encode_plus(
                    [fact for fact in left[_i:i_]],
                    max_length=self.config['padding']['inf_len'],
                    padding="max_length", truncation=True,
                    return_tensors='pt')['input_ids'].to(
                        self.device, dtype=torch.long)

                b = self.tokenizer.batch_encode_plus(
                    [fact for fact in right[_i:i_]],
                    max_length=self.config['padding']['inf_len'],
                    padding="max_length", truncation=True,
                    return_tensors='pt')['input_ids'].to(
                        self.device, dtype=torch.long)

                context = self.tokenizer.batch_encode_plus(
                    [selection["context"]
                    for selection in selections[_i:i_]],
                    max_length=self.config['padding']['sel_len'],
                    padding="max_length",
                    truncation=True,
                    return_tensors='pt')

                target = self.tokenizer.batch_encode_plus(
                    [selection["target"]
                    for selection in selections[_i:i_]],
                    max_length=self.config['padding']['inf_len'],
                    padding="max_length",
                    truncation=True,
                    return_tensors='pt')
                # process tokens and pass to the model
                ids = target['input_ids'].to(self.device, dtype=torch.long)
                y_ids = ids[:, :-1].contiguous()
                lm_labels = ids[:, 1:].clone().detach()
                lm_labels[ids[:, 1:] == 0] = -100

                input_ids = context['input_ids'].to(
                    self.device,
                    dtype=torch.long)
                mask = context['attention_mask'].to(
                    self.device,
                    dtype=torch.long)

                # Use the model to score pairs and normalize
                pair_scores = self.model(
                    input_ids=input_ids,
                    attention_mask=mask,
                    decoder_input_ids=y_ids,
                    labels=lm_labels).logits

                pair_scores = torch.log_softmax(
                    pair_scores, dim=-1).detach()

                # Leave the starting, padded tokens as it has got no score
                a = a[:, 1:]
                b = b[:, 1:]
                ids = ids[:, 1:]

                # Gather only the scores for tokens we passed to the model.
                # (rather than the entire vocab)
                pair_scores = torch.gather(
                        pair_scores, 2,
                        ids[:,:,None]).squeeze(-1)

                intersection = torch.vmap(
                    lambda t1, t2: torch.isin(
                        t1, t2))(a, b)
                intersection = torch.vmap(
                    lambda t1, t2: torch.isin(
                        t1, t2))(ids, a * intersection)

                # Get the boolean masks for special tokens in the  input, we
                # are uninterested in their logits and so we want to avoid them
                inference = ids != 21151
                selection = ids != 30575
                extras = (ids <= 32000) & (ids >= 3)
                specials = inference & selection & extras
                intersection = intersection & specials

                # Multiply intersected tokens' scores by a parameter bonus and
                # by the logarithm of their index. This index can be seen as an
                # increasing rarity, so higher indices should be more impactful
                weights = torch.where(intersection,
                    self.config['scoring']['pair_similarity']*torch.log(ids), 1)

                # If it's a positive score we multiply instead of dividing by
                # the weight, so the total always gets more positive
                pair_scores = torch.where(
                    pair_scores < 0,
                    pair_scores / weights,
                    pair_scores * weights)

                # Sum and divide by the length
                length = torch.sum(
                    pair_scores < 0, axis=1, dtype=torch.float)
                summation = torch.sum(
                    pair_scores, axis=1, dtype=torch.float)

                # Return the average score
                avg.extend(
                    torch.where(
                        length > 0,
                        summation / length,
                        summation).tolist())
                torch.cuda.empty_cache()
        return avg

    # Save the probabilities for selections in each beam
    def update_context_probabilities(self, probabilities: list):
        m = 0
        for j, question in enumerate(self.context):
            if not question["done"]:
                for k, beam in enumerate(question["beams"]):
                    self.context[j][
                        "beams"][k][
                        "score"] += probabilities[m]
                    m += 1
                self.context[j]["beams"] = sorted(self.context[j]["beams"],
                                                  key=lambda d: d["score"],
                                                  reverse=True)
                cut = self.config['search']['select_beams']
                self.context[j]["beams"] = self.context[j]["beams"][:cut]

    # Enumerate selections in each beam
    def gen_new_beams(self):
        for i, question in enumerate(self.context):
            if not question["done"]:
                new_beams = []
                for j, beam in enumerate(question["beams"]):
                    for p, a in enumerate(beam["context"]):
                        for b in beam["context"][p+1:]:
                            new_beams.append({
                                "length_of_context": beam["length_of_context"] - 2,
                                "context": [sent for sent in beam["context"]
                                            if sent != a and sent != b],
                                "selection": (a, b),
                                "score": beam["score"],
                                "proof": beam["proof"]})
                self.context[i]["beams"] = list(new_beams)

    # Generate new beams for the next iteration
    def gen_selections(self):
        self.gen_new_beams()
        probabilities = self.score_pairs()
        self.update_context_probabilities(probabilities)

    # Update the proof given an inference, and figure out if the
    # the beam entailment is done.
    def update_context_inferences(
        self,
        index: int,
        ids: torch.Tensor):
        n = 0
        for i, question in enumerate(self.context):
            if not question["done"]:
                for j, beam in enumerate(question["beams"]):
                    inference = self.tokenizer.decode(
                        ids[n],
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=True)
                    if inference[:11] == "Inference: ":
                        inference = inference[11:]
                    self.context[i]["beams"][j]["inference"] = inference
                    inference = "int" + str(index + 1) + ": " + inference
                    if beam["length_of_context"] > 0:
                        self.context[i][
                            "beams"][j][
                            "context"].append(inference)
                        self.context[i][
                            "beams"][j][
                            "length_of_context"] += 1

                    else:
                        inference = "hypothesis"
                        self.context[i]["done"] = True

                    pattern = r'sent\d{1,2}|int\d{1,2}'
                    self.context[i][
                        "beams"][j][
                        "proof"] += re.findall(
                        pattern,
                        beam["selection"][0]
                    )[0] + " & " + re.findall(
                        pattern,
                        beam["selection"][1]
                    )[0] + f" -> {inference}; "
                    n += 1

                self.context[i]["beams"] = sorted(
                    self.context[i]["beams"],
                    key=lambda d: d["score"],
                    reverse=True)
                self.context[i]["beams"] = self.context[i]["beams"][
                        :self.config['search']['entail_beams']]

                if self.context[i]["done"]:
                    self.context[i]["beams"] = self.context[i]["beams"][:1]

    # Generate inferences for our context
    def gen_inferences(self, iteration: int):
        # Generate strings for each beam's selection
        pattern = r'sent\d{1,2}: |int\d{1,2}: '
        inferences = [
            f"induce:" + " and ".join([
                f"{re.sub(pattern, '', beam['selection'][0])}",
                f"{re.sub(pattern, '', beam['selection'][1])}."])
                      for question in self.context
                      for beam in question["beams"]
                      if not question["done"]]
        pairs = [
            re.sub(pattern, '', fact)
            for question in self.context
            for beam in question["beams"]
            for fact in beam["selection"]
            if not question["done"]]
        left, right = pairs[::2], pairs[1::2]
        hypotheses = [
            question['hypothesis']
            for question in self.context
            for beam in question["beams"]
            if not question["done"]]

        # Tokenize, and initialise the logit processor, before
        # Generating an inference.
        generated_ids = []
        probabilities = []
        for _i in range(0, len(inferences), self.config['batches']['test']):
            i_ = _i + self.config['batches']['test']
            if i_ > len(inferences):
                i_ = len(inferences)
            batches = i_-_i
            data = self.tokenizer.batch_encode_plus(
                inferences[_i:i_],
                max_length=self.config['padding']['sel_len'],
                padding="max_length", truncation=True,
                return_tensors='pt')
            ids = data['input_ids'].to(self.device, dtype=torch.long)
            mask = data['attention_mask'].to(self.device, dtype=torch.long)

            h = self.tokenizer.batch_encode_plus(
                hypotheses[_i:i_],
                max_length=self.config['padding']['inf_len'],
                padding="max_length", truncation=True,
                return_tensors='pt')['input_ids'].to(
                self.device, dtype=torch.long)

            a = self.tokenizer.batch_encode_plus(
                left[_i:i_],
                max_length=self.config['padding']['inf_len'],
                padding="max_length", truncation=True,
                return_tensors='pt')['input_ids'].to(
                self.device, dtype=torch.long)

            b = self.tokenizer.batch_encode_plus(
                right[_i:i_],
                max_length=self.config['padding']['inf_len'],
                padding="max_length", truncation=True,
                return_tensors='pt')['input_ids'].to(
                    self.device, dtype=torch.long)

            diff_b = torch.vmap(lambda t1, t2: torch.isin(
                t1,t2, invert=True))(b, a) * b
            common = torch.vmap(lambda t1, t2: torch.isin(
                t1,t2))(a, b)
            diff_a = a * ~common
            pair_common = a * common
            hyp_common = torch.vmap(lambda t1, t2: torch.isin(
                t1,t2))(h, torch.cat([diff_a, diff_b], dim=1)) * h

            hypotheis_processor = DifferenceLogitProcessor(
                batches, self.config['search']['infer_beams'], hyp_common,
                self.config['logit_params']['hypothesis'])
            left_processor = DifferenceLogitProcessor(
                batches, self.config['search']['infer_beams'], diff_a,
                self.config['logit_params']['individual'])
            right_processor = DifferenceLogitProcessor(
                batches, self.config['search']['infer_beams'], diff_b,
                self.config['logit_params']['individual'])
            common_processor = DifferenceLogitProcessor(
                batches, self.config['search']['infer_beams'], pair_common,
                self.config['logit_params']['common'])

            generated_ids.extend(self.model.generate(
                input_ids = ids,
                attention_mask = mask,
                max_length = self.config['padding']['inf_len'],
                min_length = 8,
                repetition_penalty = self.config['scoring']['repetition_penalty'],
                num_beams = self.config['search']['infer_beams'],
                num_return_sequences = self.config['search']['infer_beams'],
                logits_processor = [
                    hypotheis_processor,
                    left_processor,
                    right_processor,
                    common_processor]))

        if self.config['logging']['inferences']:
            self.debug_inferences(generated_ids)
            torch.cuda.empty_cache()
        self.update_context_inferences(iteration, generated_ids[
            ::self.config['search']['infer_beams']])

    # Printing inferences for debugging
    def debug_inferences(self, generated_ids):
        print("\n\n")
        n = 0
        for i, question in enumerate(self.context):
            if not question["done"]:
                print(f'{question["hypothesis"]}\n{"="*100}')
                for j, beam in enumerate(question["beams"]):
                    [print(f'- {fact}') for fact in beam["selection"]]
                    print(f'~ score: {beam["score"]}')
                    print("-"*100)
                    [print(f'>   {tokens}')
                    for tokens in self.tokenizer.batch_decode(
                        generated_ids[
                            n:n+self.config['search']['infer_beams']],
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=True
                        )]
                    print("+"*100)
                    n+=self.config['search']['infer_beams']
                print("\n")

# Train
Load wandb, set the config settings, then load the dataset and begin training

In [ ]:
wandb.init(project="Individual-Project",settings=wandb.Settings(start_method="thread"))

parameters = {
    'training': {
        'epochs': 4,
        'eval': 10},
    'batches': {
        'train': 32,
        'test': 8},
    'padding': {
        'sel_len': 380,
        'inf_len': 100},
    'search': {
        'iterations': 1,
        'entail_beams': 6,
        'select_beams': 12,
        'infer_beams': 12},
    'scoring': {
        'pair_similarity': 2.5,
        'repetition_penalty': 24.0},
    'logit_params': {
        'hypothesis': {
            'magnitude': 2.0,
            'decay': 4.2,
            'threshold': 0.15,
            'steps': 6},
        'individual': {
            'magnitude': 2.2,
            'decay': 3.0,
            'threshold': 0.34,
            'steps': 3},
        'common': {
            'magnitude': 1.1,
            'decay': 0,
            'threshold': 0.5,
            'steps': 1}},
    'logging': {
        'verbose': False,
        'inferences': True,
        'save_file': False,
        'frequencies': False,
        'vocab': False},
    'directory': {}}

# WandB – Config is a variable that holds and saves hyperparameters and inputs
# Defining some key variables that will be used later on in the training
config = wandb.config  # Initialize config
config.TRAIN_BATCH_SIZE = parameters['batches']['train']  # input batch size for training (default: 64)
config.VALID_BATCH_SIZE = parameters['batches']['test']  # input batch size for testing (default: 1000)
config.TRAIN_EPOCHS = parameters['training']['epochs']  # number of epochs to train (default: 10)
config.VAL_EPOCHS = 1
config.LEARNING_RATE = 1e-3  # learning rate (default: 0.01)
config.SEED = 42  # random seed (default: 42)

parameters['directory']['runs'] = "predictions"
parameters['directory']['data'] = "public_dataset"

parameters['directory']['size'] = "small"
parameters['directory']['method'] = "beam"
parameters['directory']['version'] = "v1"

# Upload Files
data = parameters['directory']['data']
if not os.path.isdir(data):
    os.mkdir(data)
data_method = f"{data}/{parameters['directory']['method']}"
if not os.path.isdir(data_method):
    os.mkdir(data_method)
if not all([
    os.path.isfile(f'{data_method}/selection_train.jsonl'),
    os.path.isfile(f'{data_method}/inference_train.jsonl'),
    os.path.isfile(f'{data_method}/test.jsonl')]):
    uploaded = files.upload()
    for filename in uploaded.keys():
      dst_path = os.path.join(f'{data_method}', filename)
      print(f'move {filename} to {dst_path}')
      shutil.move(filename, dst_path)

runs = parameters['directory']['runs']
if not os.path.isdir(runs):
    os.mkdir(runs)

runs_method = f"{runs}/{parameters['directory']['method']}"
if not os.path.isdir(runs_method):
    os.mkdir(runs_method)

version = f"{runs_method}/{parameters['directory']['version']}"
if not os.path.isdir(version):
    os.mkdir(version)
    os.mkdir(f'{version}/trees')

with open(f'{version}/parameters.json', 'w') as file:
    hyps = json.dumps(parameters)
    file.write(hyps+'\n')

# Set random seeds and deterministic pytorch for reproducibility
torch.manual_seed(config.SEED)  # pytorch random seed
np.random.seed(config.SEED)  # numpy random seed
torch.backends. cudnn.deterministic = True

# tokenzier for encoding the text
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# Importing and Pre-Processing the domain data
# Selecting the needed columns only.
# Adding the summarzie text in front of the text. This is to format the dataset similar to how T5 model was trained for summarization task.
selection_set = pd.read_json(f'{data_method}/selection_train.jsonl', encoding='latin-1', lines=True)
selection_set = pd.DataFrame({
    'context': selection_set['context'],
    'target': selection_set['target']})

inference_set = pd.read_json(f'{data_method}/inference_train.jsonl', encoding='latin-1', lines=True)
inference_set = pd.DataFrame({
    'context': inference_set['context'],
    'target': inference_set['target']})

test = pd.read_json(f'{data_method}/test.jsonl', encoding='latin-1', lines=True)
test = test[['context', 'question', 'proof', 'hypothesis']]
test = pd.DataFrame({
    'context': 'question: ' + test['question'] + ' hypothesis: ' + test['hypothesis'] + ' ' + test[
        'context'],
    'target': test['proof']})
print("Selection training dataset:")
print(selection_set.head())
print("Inference training dataset:")
print(inference_set.head())

# Creation of Dataset
train_dataset = pd.concat(
    [selection_set,
     inference_set],
    ignore_index = True)

eval_dataset = test

print("TRAIN Dataset: {}".format(train_dataset.shape))
print(train_dataset.head())

print("TEST Dataset: {}".format(eval_dataset.shape))

# Creating the Training and Validation dataset for further creation of Dataloader
training_set = TokenizedDataset(train_dataset, tokenizer, parameters)
eval_set = eval_dataset

# Defining the model. We are using t5-base model and added a Language model layer on top for generation of Summary.
# Further this model is sent to device (GPU/TPU) for using the hardware.
model = T5ForConditionalGeneration.from_pretrained("t5-small")

model = model.to(device)

training_args = TrainingArguments(
    output_dir="checkpoints",
    save_strategy="steps",
    save_steps=10000,
    save_total_limit=1,
    evaluation_strategy="steps",
    eval_steps=parameters['training']['eval'],
    max_steps=parameters['training']['eval'],
    learning_rate=1e-3,
    per_device_train_batch_size=parameters['batches']['train'],
    per_device_eval_batch_size=parameters['batches']['test'],
    seed=42)

trainer = CustomTrainer(
    trainer_model=model,
    args=training_args,
    trainer_train_dataset=training_set,
    trainer_eval_dataset=eval_set,
    trainer_tokenizer=tokenizer,
    trainer_device=device,
    trainer_config=parameters)

trainer.train()

Selection training dataset:
                                             context  \
0  correlate: Stars are organized into patterns c...   
1  correlate: Stars are organized into patterns c...   
2  correlate: Which of the following statements b...   
3  correlate: Which of the following statements b...   
4  correlate: From Earth, the Sun appears brighte...   

                                              target  
0  leo is a kind of constellation and a constella...  
1  leo is a constellation containing stars and th...  
2  diurnal motion is when objects in the sky appe...  
3  stars apearing to move relative to the horizon...  
4  a star produces light and a source of somethin...  
Inference training dataset:
                                             context  \
0  induce: leo is a kind of constellation and a c...   
1  induce: diurnal motion is when objects in the ...   
2  induce: a star produces light and a source of ...   
3  induce: stars are a source of light and as a s... 

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


OutOfMemoryError: ignored

# Utils
A few utility functions for designing minimum/maximum values, to make the code faster, and so cheaper.

In [ ]:
parameters['logging']['vocab'] = False
# Used to inform which tokens are disproportionately represented in the
# dataset, and so are not worth giving a bonus when they are within the
# intersection of a pair of facts
def token_frequencies(data: TokenizedDataset, batch_size):
    loader = DataLoader(data, batch_size = batch_size)
    tokens = []
    for item in loader:
        ids = item["input_ids"]
        y = item["decoder_input_ids"]
        tokens.extend(ids)
        tokens.extend(y)

    tokens = torch.cat(tokens)
    uniques, counts = tokens.unique(return_counts=True)
    for i, token in enumerate(uniques):
        print(i) if i % 512 == 0 else None
        string = tokenizer.decode(uniques[i])
        token = str(uniques[i].item())
        count = str(counts[i].item())
        string_length = len(string)
        token_length = len(token)
        count_length = len(count)
        print(
            f'"{string}" {" "*(20-string_length)} ({token}) {" " * (5-token_length)} {count}'
        )
        print("-"*100) if i+1 % 512 == 0 else None

if parameters['logging']['vocab']:
    token_frequencies(training_set, config.TRAIN_BATCH_SIZE)

In [ ]:
# Used to find the maximum length of inputs and outputs
def max_lens(eval_dataset: RawDataset, batch_size, tokenizer):
    context = eval_dataset.get_context_dictionary(batch_size)
    selections, inferences, approx_targets = [], [], []
    pattern = r'sent\d{1,2}|int\d{1,2}'
    for question in context:
        for beam in question["beams"]:
            for i, a in enumerate(beam["context"]):
                for b in beam["context"][i+1:]:
                    fact_string = '. _____ '.join([
                        re.sub(pattern, '', sent)
                        for sent in beam['context']]
                    )
                    selections.append({
                        "context": f"correlate {question['preface']}. _____ {fact_string}.",
                        "target": "".join(
                            [f" _____ {re.sub(pattern, '', a)},",
                            f" _____ {re.sub(pattern, '', b)}."
                            ])
                    })
                    inferences.append(
                        f"induce" + "".join([
                            f" _____ {re.sub(pattern, '', a)}.",
                            f" _____ {re.sub(pattern, '', b)}."
                            f" _____ "
                        ])
                    )
                    approx_targets.append(
                        f"{re.sub(pattern, '', a)} and {re.sub(pattern, '', b)}"
                    )

    _max_sel, max_sel_, _max_inf, max_inf_ = 0, 0, 0, 0
    n_questions = len(selections)
    for i in range(n_questions):

        _max_s = max([tokenizer(
            selections[i]["context"],
            return_tensors='pt'
        )["input_ids"].shape[1]])

        max_s_ = max([tokenizer(
            selections[i]["target"],
            return_tensors='pt'
        )["input_ids"].shape[1]])

        _max_i = max([tokenizer(
            inferences[i],
            return_tensors='pt'
        )["input_ids"].shape[1]])

        max_i_ = max([tokenizer(
            approx_targets[i],
            return_tensors='pt'
        )["input_ids"].shape[1]])

        if _max_s > _max_sel:
            _max_sel = _max_s
            __select = selections[i]["context"]
        if max_s_ > max_sel_:
            max_sel_ = max_s_
            select__ = selections[i]["target"]
        if _max_i > _max_inf:
            _max_inf = _max_i
            ___infer = inferences[i]
        if max_i_ > max_inf_:
            max_inf_ = max_i_
            infer___ = approx_targets[i]

    maxima = (
        (
            (
                __select,
                _max_sel
            ),
            (
                select__,
                max_sel_
            )
        ),
        (
            (
                ___infer,
                _max_inf
            ),
            (
                infer___,
                max_inf_
            )
        )
    )
    prefix = ["selection", "inference"]
    in_out = ["input", "target"]
    for i in range(2):
        print(f'{prefix[i]}: ')
        for j in range(2):
            print(f'  {in_out[j]}:')
            print('    text: ', end='')
            for k, c in enumerate(maxima[i][j][0]):
                print(c,end='')
                if ((k+1) % 80) == 0:
                    print("-\n          ", end='')
            print(f'    \nsize: {maxima[i][j][1]}')

if parameters['logging']['frequencies']:
    max_lens(
        RawDataset(eval_set),
        config.TRAIN_BATCH_SIZE,
        tokenizer)